## Grid Search for Analyzer Settings

Re-runs PubTrends clustering with different weights for the following measures:
 * bibliographic coupling (BC)
 * co-citations (CC)
 * direct citations (DC)
 * text citations (TC)

In [ ]:
import json
import itertools
import logging
import os
import pandas as pd

from sklearn.metrics.cluster import adjusted_mutual_info_score, v_measure_score
from sklearn.model_selection import ParameterGrid

from pysrc.papers.analysis.graph import node2vec
from pysrc.papers.analyzer import PapersAnalyzer
from pysrc.papers.config import AnalyzerSettings

from utils.analysis import get_direct_references_subgraph, align_clusterings_for_sklearn, topic_analysis_node2vec
from utils.io import reload_exported_analyzer, load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
OUTPUT_NAME = 'grid_search_node2vec_lda_louvain_2021-06-27.csv'

## Clustering Methods

#### Latent Dirichlet Allocation

In [ ]:
from collections import Counter
from functools import lru_cache

from pysrc.papers.analysis.text import build_corpus

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer


def cluster_lda(X, min_cluster_size, max_clusters):
    logging.debug('Looking for an appropriate number of clusters,'
                 f'min_cluster_size={min_cluster_size}, max_clusters={max_clusters}')
    r = min(max_clusters, X.shape[0]) + 1
    l = 1

    if l >= r - 2:
        return [0] * X.shape[0], None

    while l < r - 2:
        n_clusters = int((l + r) / 2)
        lda = LatentDirichletAllocation(n_components=n_clusters, random_state=0).fit(X)
        clusters = lda.transform(X).argmax(axis=1)
        clusters_counter = Counter(clusters)
        min_size = clusters_counter.most_common()[-1][1]
        if min_size < min_cluster_size:
            r = n_clusters + 1
        elif min_size > max_clusters:
            l = n_clusters
        else:
            break

    logging.debug(f'Number of clusters = {n_clusters}')
    logging.debug('Reorder clusters by size descending')
    reorder_map = {c: i for i, (c, _) in enumerate(clusters_counter.most_common())}
    return [reorder_map[c] for c in clusters]


@lru_cache(maxsize=1000)
def preproc_lda(analyzer, subgraph, max_df, min_df, max_features):
    # Use only papers present in the subgraph
    subgraph_df = analyzer.df[analyzer.df.id.isin(subgraph.nodes)]
    node_ids = list(subgraph_df.id.values)
    
    # Build and vectorize corpus for LDA
    corpus = build_corpus(subgraph_df)
    vectorizer = CountVectorizer(max_df=max_df, min_df=min_df, max_features=max_features)
    X = vectorizer.fit_transform(corpus)
    return X, node_ids
    

def topic_analysis_lda(analyzer, subgraph, **settings):
    X, node_ids = preproc_lda(analyzer, subgraph, 
                              max_df=settings['max_df'],
                              min_df=settings['min_df'],
                              max_features=settings['max_features'])
    
    clusters = cluster_lda(X, min_cluster_size=settings['topic_min_size'],
                           max_clusters=settings['topics_max_number'])
    
    return dict(zip(node_ids, clusters))

#### Louvain Community Clustering (restored from Git history)

In [ ]:
import community
import numpy as np
import networkx as nx

from pysrc.papers.utils import SEED

def topic_analysis_louvain(analyzer, similarity_graph, **settings):
    """
    Performs clustering of similarity topics, merging small topics into Other component
    :param similarity_graph: Similarity graph
    :param similarity_func: Function to compute hybrid similarity between nodes in similarity graph
    :param topic_min_size:
    :param max_topics_number:
    :return: merged_partition, comp_dendrogram, comp_sizes
    """
    connected_components = nx.number_connected_components(similarity_graph)
    logging.debug(f'Similarity graph has {connected_components} connected components')

    logging.debug('Compute aggregated similarity')
    similarity_func = PapersAnalyzer.similarity
    for _, _, d in similarity_graph.edges(data=True):
        d['similarity'] = similarity_func(d)

    logging.debug('Graph clustering via Louvain community algorithm')
    partition_louvain = community.best_partition(
        similarity_graph, weight='similarity', random_state=SEED
    )
    logging.debug(f'Best partition {len(set(partition_louvain.values()))} components')
    components = set(partition_louvain.values())
    comp_sizes = {c: sum([partition_louvain[node] == c for node in partition_louvain.keys()]) for c in components}
    logging.debug(f'Components: {comp_sizes}')

    if len(similarity_graph.edges) > 0:
        modularity = community.modularity(partition_louvain, similarity_graph)
        logging.debug(f'Graph modularity (possible range is [-1, 1]): {modularity :.3f}')

    logging.debug('Merge small components')
    similarity_matrix = compute_similarity_matrix(similarity_graph, similarity_func, partition_louvain)
    merged_partition, comp_sizes = merge_components(
        partition_louvain, similarity_matrix,
        topic_min_size=settings['topic_min_size'], max_topics_number=settings['topics_max_number']
    )

    logging.debug('Compute partition the Louvain algorithm')
    dendrogram = community.generate_dendrogram(
        similarity_graph, weight='similarity', random_state=SEED
    )

    logging.debug('Update components partition according to merged components')
    if len(dendrogram) >= 2:
        rename_map = {}
        for pid, c in merged_partition.items():
            rename_map[dendrogram[0][pid]] = c
        comp_level = {rename_map[k]: v for k, v in dendrogram[1].items()}
        comp_dendrogram = cleanup_dendrogram([comp_level] + dendrogram[2:])
    else:
        comp_dendrogram = []

    return merged_partition #, comp_dendrogram, comp_sizes


def compute_similarity_matrix(similarity_graph, similarity_func, partition):
    logging.debug('Computing mean similarity of all edges between topics')
    n_comps = len(set(partition.values()))
    edges = len(similarity_graph.edges)
    sources = [None] * edges
    targets = [None] * edges
    similarities = [0.0] * edges
    i = 0
    for u, v, data in similarity_graph.edges(data=True):
        sources[i] = u
        targets[i] = v
        similarities[i] = similarity_func(data)
        i += 1
    df = pd.DataFrame(partition.items(), columns=['id', 'comp'])
    similarity_df = pd.DataFrame(data={'source': sources, 'target': targets, 'similarity': similarities})
    similarity_topics_df = similarity_df.merge(df, how='left', left_on='source', right_on='id') \
        .merge(df, how='left', left_on='target', right_on='id')
    logging.debug('Calculate mean similarity between for topics')
    mean_similarity_topics_df = \
        similarity_topics_df.groupby(['comp_x', 'comp_y'])['similarity'].mean().reset_index()
    similarity_matrix = np.zeros(shape=(n_comps, n_comps))
    for index, row in mean_similarity_topics_df.iterrows():
        cx, cy = int(row['comp_x']), int(row['comp_y'])
        similarity_matrix[cx, cy] = similarity_matrix[cy, cx] = row['similarity']
    return similarity_matrix


def merge_components(partition, similarity_matrix, topic_min_size, max_topics_number):
    """
    Merge small topics to required number of topics and minimal size, reorder topics by size
    :param partition: Partition, dict paper id -> component
    :param similarity_matrix: Mean similarity between components for partition
    :param topic_min_size: Min number of papers in topic
    :param max_topics_number: Max number of topics
    :return: merged_partition, sorted_comp_sizes
    """
    logging.debug(f'Merging: max {max_topics_number} components with min size {topic_min_size}')
    comp_sizes = {c: sum([partition[paper] == c for paper in partition.keys()])
                  for c in (set(partition.values()))}
    logging.debug(f'{len(comp_sizes)} comps, comp_sizes: {comp_sizes}')

    merge_index = 1
    while len(comp_sizes) > 1 and \
            (len(comp_sizes) > max_topics_number or min(comp_sizes.values()) < topic_min_size):
        logging.debug(f'{merge_index}. Pick minimal and merge it with the closest by similarity')
        merge_index += 1
        min_comp = min(comp_sizes.keys(), key=lambda c: comp_sizes[c])
        comp_to_merge = max([c for c in partition.values() if c != min_comp],
                            key=lambda c: similarity_matrix[min_comp][c])
        logging.debug(f'Merging with most similar comp {comp_to_merge}')
        comp_update = min(min_comp, comp_to_merge)
        comp_sizes[comp_update] = comp_sizes[min_comp] + comp_sizes[comp_to_merge]
        if min_comp != comp_update:
            del comp_sizes[min_comp]
        else:
            del comp_sizes[comp_to_merge]
        logging.debug(f'Merged comps: {len(comp_sizes)}, updated comp_sizes: {comp_sizes}')
        for (paper, c) in list(partition.items()):
            if c == min_comp or c == comp_to_merge:
                partition[paper] = comp_update

        logging.debug('Update similarities')
        for i in range(len(similarity_matrix)):
            similarity_matrix[i, comp_update] = \
                (similarity_matrix[i, min_comp] + similarity_matrix[i, comp_to_merge]) / 2
            similarity_matrix[comp_update, i] = \
                (similarity_matrix[min_comp, i] + similarity_matrix[comp_to_merge, i]) / 2

    logging.debug('Sorting comps by size descending')
    sorted_components = dict(
        (c, i) for i, c in enumerate(sorted(set(comp_sizes), key=lambda c: comp_sizes[c], reverse=True))
    )
    logging.debug(f'Comps reordering by size: {sorted_components}')
    merged_partition = {paper: sorted_components[c] for paper, c in partition.items()}
    sorted_comp_sizes = {c: sum([merged_partition[p] == c for p in merged_partition.keys()])
                         for c in set(merged_partition.values())}

    for k, v in sorted_comp_sizes.items():
        logging.debug(f'Component {k}: {v} ({int(100 * v / len(merged_partition))}%)')
    return merged_partition, sorted_comp_sizes


def cleanup_dendrogram(dendrogram):
    """Remove redundant levels from the partition"""
    rd = []
    for i, level in enumerate(dendrogram):
        if i == 0:
            rd.append(level)
        else:
            # Omit redundant level
            if len(set(level.keys())) == len(set(level.values())):
                rd[i - 1] = {k: level[v] for k, v in rd[i - 1].items()}
            else:
                rd.append(level)
    return rd

#### Topic Analysis Launcher (node2vec / louvain / LDA)

In [ ]:
def topic_analysis(analyzer, subgraph, method, **method_params):
    """
    Returns partition - dictionary {pmid (str): cluster (int)}
    """
    if method == 'node2vec':
        return topic_analysis_node2vec(analyzer, subgraph, **method_params)
    elif method == 'lda':
        return topic_analysis_lda(analyzer, subgraph, **method_params)
    elif method == 'louvain':
        return topic_analysis_louvain(analyzer, subgraph, **method_params)
    else:
        raise ValueError('Unknown clustering method')

## Grid Search

In [ ]:
def run_grid_search(analyzer, subgraph, ground_truth, metrics, target_metric, param_grid, save_partition=False):
    # Accumulate grid results for all hierarchy levels
    grid_results = []
    partitions = []

    for param_values in ParameterGrid(param_grid):
        partition = topic_analysis(analyzer, subgraph, **param_values)

        if save_partition:
            param_partition = param_values.copy()
            param_partition['partition'] = partition
            partitions.append(param_partition)

        # Iterate over hierarchy levels to avoid re-calculating same clustering for different ground truth    
        for level, ground_truth_partition in ground_truth.items():
            result = param_values.copy()
            result['level'] = level
            labels_true, labels_pred = align_clusterings_for_sklearn(partition, ground_truth_partition)
            result['n_clusters'] = len(set(labels_pred))

            # Save partition & iterate over different metrics
            for metric in metrics:
                result[metric.__name__] = metric(labels_true, labels_pred)

            grid_results.append(result)
        print('.', end='')
    print('\n')
    
    return grid_results, partitions

## Main Part

* Setting parameter grid
* Choosing metrics to be evaluated
* Running loop over all review papers

In [ ]:
param_grid = [
    {
        'method': ['node2vec'],
        'walk_length': [10, 50, 100],
        'walks_per_node': [10, 20],
        'vector_size': [32, 64],
        'topic_min_size': [10, 20],
        'topics_max_number': [10, 20, 30],
    },
    {
        'method': ['louvain'],
        'topic_min_size': [10, 20],
        'topics_max_number': [10, 20, 30],
    },
    {
        'method': ['lda'],
        'max_df': [0.8],
        'min_df': [0.001],
        'max_features': [10000],
        'topic_min_size': [10, 20],
        'topics_max_number': [10, 20, 30],
    }
]

In [ ]:
metrics = [adjusted_mutual_info_score, pd_score, reg_v_score]
target_metric = pd_score

In [ ]:
results_df = pd.DataFrame()

for pmid in get_review_pmids():
    clustering = load_clustering(pmid)
    analyzer = load_analyzer(pmid)
    logging.info(f'{pmid} - loaded clustering and analyzer')
    logging.info(f'{pmid} - rebuilt similarity graph with scaling')
    
    # Pre-calculate all hierarchy levels before grid search to avoid re-calculation of clusterings
    ground_truth = {}
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[level] = preprocess_clustering(clustering, level, 
                                                    include_box_sections=False,
                                                    uniqueness_method='unique_only')
    logging.info(f'{pmid} - calculated ground truth for all hierarchy levels')
    
    subgraph = get_direct_references_subgraph(analyzer, pmid)
    logging.info(f'{pmid} - running grid search')
    
    grid_results, partitions = run_grid_search(analyzer, subgraph, ground_truth, 
                                               metrics, target_metric, param_grid)
    
    grid_results_df = pd.DataFrame(grid_results)
    grid_results_df['pmid'] = pmid
    results_df = results_df.append(grid_results_df, ignore_index = True)
    
    logging.info(f'{pmid} - done')

In [ ]:
results_df = results_df.fillna(0)

In [ ]:
results_df.to_csv(OUTPUT_NAME, index=False)

In [ ]:
print('Grid Search - Done')

## Simple Visualization

#### Load old results if needed

In [ ]:
load_from_disk = False

In [ ]:
if load_from_disk:
    results_df = pd.read_csv(OUTPUT_NAME)

#### Extract parameter columns

In [ ]:
score_columns = set([m.__name__ for m in metrics])
param_columns = list(set(results_df.columns) - score_columns - set(['level', 'n_clusters', 'pmid']))
print(param_columns)

#### Average Scores 

In [ ]:
def get_top_parameter_sets_for_method(score_df, param_cols, method, target_col, n=5):
    return score_df[score_df.method == method].groupby(param_cols)[[target_col, 'n_clusters']].mean().sort_values(by=target_col, 
                                                                                                                  ascending=False).head(n).reset_index()

In [ ]:
def get_top_mean_score_for_method(score_df, param_cols, method, target_col):
    return score_df[score_df.method == method].groupby(param_cols)[target_col].mean().sort_values(ascending=False).values[0]

In [ ]:
target_col = metrics[0].__name__

for method in results_df.method.unique():
    print(col, get_top_mean_score_for_method(results_df, param_columns, method, target_col), '\n')
    print(get_top_parameter_sets_for_method(results_df, param_columns, method, target_col), '\n')

In [ ]:
mean_score_df = score_df.groupby('method').max().reset_index()
mean_score_df.head()

In [ ]:
mean_score_df.plot(x='method', y=list(score_columns))

#### Average Scores for Different Clustering Levels

In [ ]:
level_score_df = results_df.groupby(param_columns + ['level'])[list(score_columns) + ['n_clusters']].mean().reset_index()
level_score_df.head(10)

In [ ]:
mean_level_score_df = level_score_df.groupby(['method', 'level'])[target_metric.__name__].max().reset_index()
mean_level_score_df.head(10)

In [ ]:
for level in mean_level_score_df.level.unique():
    mean_level_score_df[mean_level_score_df.level == level].plot(x='method', y=target_metric.__name__)

In [ ]:
print('Visualization - Done')